# SISTEMA DE ALERTAS: Riesgo de Tardanza al Despacho

**Objetivo operativo:** Generar alertas de riesgo de tardanza para envíos en el momento del despacho, basado en el Modelo 1 de clasificación.

**Política de alertas:** Configurable entre umbral fijo o cupo top-k (capacidad operativa limitada).

**Output:** CSV con alertas + resumen ejecutivo para acción inmediata.

---

## SECCIÓN 1 — CONFIGURACIÓN OPERATIVA

Parámetros configurables del sistema de alertas.

**Cuándo usar cada política:**
- **Threshold (0.5)**: Cuando se pueden revisar todas las alertas que salgan, sin límite de capacidad.
- **Top-k**: Cuando la capacidad operativa es limitada (ej: solo podemos revisar el 20% de envíos más riesgosos).

---

In [1]:
# ========== PARÁMETROS CONFIGURABLES ==========

# Política de alertas: "threshold" o "topk"
alert_policy = "topk"  # Cambiar a "threshold" para umbral fijo

# Si policy = "threshold"
threshold_value = 0.5  # Alertar si score >= 0.5

# Si policy = "topk"
topk_percent = 20  # Alertar top 20% de scores más altos

# Archivos de salida
output_alerts_file = "alertas.csv"
output_scoring_full = "scoring_completo.csv"
output_summary_file = "resumen_ejecutivo.txt"

# Modo simulación (usar test histórico) o archivo nuevo
use_production_file = False  # Cambiar a True si hay "nuevos_despachos.csv"
production_file = "nuevos_despachos.csv"

print('='*70)
print(' '*15 + 'CONFIGURACIÓN DEL SISTEMA')
print('='*70)
print(f'\n📋 Parámetros:')
print(f'   Política: {alert_policy.upper()}')
if alert_policy == "threshold":
    print(f'   Umbral: {threshold_value}')
elif alert_policy == "topk":
    print(f'   Cupo: Top {topk_percent}%')
print(f'\n📁 Archivos de salida:')
print(f'   - {output_alerts_file}')
print(f'   - {output_scoring_full}')
print(f'   - {output_summary_file}')
print(f'\n✓ Configuración cargada')

               CONFIGURACIÓN DEL SISTEMA

📋 Parámetros:
   Política: TOPK
   Cupo: Top 20%

📁 Archivos de salida:
   - alertas.csv
   - scoring_completo.csv
   - resumen_ejecutivo.txt

✓ Configuración cargada


## Imports

Librerías necesarias para el sistema de alertas.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Preprocesamiento
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

# Modelo
try:
    import lightgbm as lgb
except ModuleNotFoundError:
    import sys
    import subprocess
    print('Instalando lightgbm...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lightgbm'])
    import lightgbm as lgb

# Métricas (solo para validación en simulación)
from sklearn.metrics import precision_score, recall_score

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('✓ Librerías cargadas')
print(f'LightGBM version: {lgb.__version__}')

✓ Librerías cargadas
LightGBM version: 4.6.0


---

## SECCIÓN 2 — CARGA Y PREPARACIÓN DEL UNIVERSO

Cargar datos, filtrar universo válido (excluir cancelados) y definir ventana temporal para alertas.

---

### Cargar Dataset Limpio

Cargar el dataset limpio generado en el Paso 2.

In [3]:
print('='*70)
print(' '*15 + 'CARGA DE DATOS')
print('='*70)

data_dir = Path('../data/processed')

# Cargar dataset limpio
df = pd.read_csv(data_dir / 'DataCoSupplyChainDataset_clean.csv')

print(f'\n✓ Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'  Fecha rango potencial: {df.shape[0]:,} registros')

               CARGA DE DATOS

✓ Dataset cargado: 180,519 filas × 35 columnas
  Fecha rango potencial: 180,519 registros


### Preparación: Parseo de Fecha y Filtros

Parsear fecha de despacho y filtrar envíos cancelados (universo consistente).

In [4]:
print('\n=== PREPARACIÓN DE DATOS ===\n')

# Parsear shipping_date
df['shipping_date_(DateOrders)'] = pd.to_datetime(df['shipping_date_(DateOrders)'], errors='coerce')
nulos_fecha = df['shipping_date_(DateOrders)'].isna().sum()

if nulos_fecha > 0:
    print(f'⚠️ Eliminando {nulos_fecha:,} filas sin fecha válida')
    df = df.dropna(subset=['shipping_date_(DateOrders)'])

print(f'✓ Fechas parseadas: {df["shipping_date_(DateOrders)"].min().date()} a {df["shipping_date_(DateOrders)"].max().date()}')

# Filtrar cancelados
if 'Delivery_Status' in df.columns:
    filas_antes = len(df)
    df = df[~df['Delivery_Status'].str.contains('cancel', case=False, na=False)].copy()
    print(f'✓ Filtro cancelados: {filas_antes:,} → {len(df):,} filas')
else:
    print('⚠️ Columna Delivery_Status no encontrada')

print(f'\n✓ Dataset preparado: {len(df):,} filas')


=== PREPARACIÓN DE DATOS ===

✓ Fechas parseadas: 2015-01-03 a 2018-02-06
✓ Filtro cancelados: 180,519 → 172,765 filas

✓ Dataset preparado: 172,765 filas


### Definir Ventana de Alertas (Simulación o Producción)

En modo simulación usamos el período TEST histórico. En producción se cargaría un archivo de nuevos despachos.

In [5]:
print('\n=== DEFINICIÓN DE VENTANA DE ALERTAS ===\n')

if use_production_file:
    # Modo PRODUCCIÓN: cargar archivo nuevo
    prod_path = data_dir / production_file
    if prod_path.exists():
        df_alertar = pd.read_csv(prod_path)
        df_alertar['shipping_date_(DateOrders)'] = pd.to_datetime(df_alertar['shipping_date_(DateOrders)'], errors='coerce')
        print(f'✓ Archivo de producción cargado: {prod_path}')
        print(f'  Registros a evaluar: {len(df_alertar):,}')
        modo = "PRODUCCIÓN"
    else:
        print(f'❌ Archivo {production_file} no encontrado')
        print(f'   Cambiando a modo SIMULACIÓN...')
        use_production_file = False

if not use_production_file:
    # Modo SIMULACIÓN: usar período TEST del split histórico
    print(f'📊 Modo SIMULACIÓN activado')
    print(f'   Usando período TEST del split temporal (últimos 20% de datos)')
    
    # Ordenar por fecha
    df_sorted = df.sort_values('shipping_date_(DateOrders)').reset_index(drop=True)
    
    # Split 80/20
    split_idx = int(len(df_sorted) * 0.8)
    df_train = df_sorted.iloc[:split_idx].copy()
    df_alertar = df_sorted.iloc[split_idx:].copy()
    
    print(f'\n   TRAIN (para modelo): {len(df_train):,} filas')
    print(f'     Fecha: {df_train["shipping_date_(DateOrders)"].min().date()} → {df_train["shipping_date_(DateOrders)"].max().date()}')
    
    print(f'\n   TEST (para alertas simuladas): {len(df_alertar):,} filas')
    print(f'     Fecha: {df_alertar["shipping_date_(DateOrders)"].min().date()} → {df_alertar["shipping_date_(DateOrders)"].max().date()}')
    
    modo = "SIMULACIÓN"

# Crear row_id estable
df_alertar = df_alertar.reset_index(drop=True)
df_alertar['row_id'] = 'alert_' + df_alertar.index.astype(str)

print(f'\n✓ Ventana de alertas definida: {len(df_alertar):,} registros ({modo})')
print(f'✓ row_id creado para trazabilidad')


=== DEFINICIÓN DE VENTANA DE ALERTAS ===

📊 Modo SIMULACIÓN activado
   Usando período TEST del split temporal (últimos 20% de datos)

   TRAIN (para modelo): 138,212 filas
     Fecha: 2015-01-03 → 2017-04-25

   TEST (para alertas simuladas): 34,553 filas
     Fecha: 2017-04-25 → 2018-02-06

✓ Ventana de alertas definida: 34,553 registros (SIMULACIÓN)
✓ row_id creado para trazabilidad


---

## SECCIÓN 3 — SCORING (Probabilidad de Tardanza)

Generar score de riesgo para cada envío. Re-entrenar modelo si no existe modelo guardado.

---

### Verificar Modelo Guardado

Intentar cargar modelo serializado. Si no existe, re-entrenar.

In [6]:
print('='*70)
print(' '*15 + 'GENERACIÓN DE SCORES')
print('='*70)

# Buscar modelo guardado
model_file = data_dir / 'modelo1.pkl'
model_exists = model_file.exists()

print(f'\n🔍 Buscando modelo guardado: {model_file}')
if model_exists:
    print(f'   ✓ Modelo encontrado - cargando...')
    # Aquí iría código de carga (pickle/joblib)
    # Por ahora asumimos que no existe
    model_exists = False
    print(f'   ⚠️ Carga de modelo no implementada en esta versión')
    print(f'   → Re-entrenando modelo...')
else:
    print(f'   ⚠️ Modelo no encontrado')
    print(f'   → Re-entrenando modelo desde cero...')

               GENERACIÓN DE SCORES

🔍 Buscando modelo guardado: ..\data\processed\modelo1.pkl
   ⚠️ Modelo no encontrado
   → Re-entrenando modelo desde cero...


### Re-entrenar Modelo 1 (Si es necesario)

Entrenar LightGBM con las mismas features e hiperparámetros del Modelo 1 original.

In [7]:
if not model_exists and modo == "SIMULACIÓN":
    print('\n=== RE-ENTRENAMIENTO DEL MODELO 1 ===\n')
    
    # Features del Modelo 1
    numeric_features = [
        'Days_for_shipment_(scheduled)',
        'processing_time',
        'Order_Item_Quantity',
        'Sales',
        'Order_Item_Product_Price',
        'Order_Profit_Per_Order',
        'Product_Price'
    ]
    
    categorical_features = [
        'Shipping_Mode',
        'Order_Region',
        'Market',
        'Order_Country',
        'Order_State',
        'Order_City',
        'Category_Name',
        'Department_Name',
        'Customer_Segment'
    ]
    
    # Filtrar features que existen
    numeric_features = [f for f in numeric_features if f in df_train.columns]
    categorical_features = [f for f in categorical_features if f in df_train.columns]
    
    print(f'📋 Features del modelo:')
    print(f'   Numéricas: {len(numeric_features)}')
    print(f'   Categóricas: {len(categorical_features)}')
    
    # Crear features temporales
    for df_temp in [df_train, df_alertar]:
        df_temp['shipping_month'] = df_temp['shipping_date_(DateOrders)'].dt.month
        df_temp['shipping_dayofweek'] = df_temp['shipping_date_(DateOrders)'].dt.dayofweek
        df_temp['shipping_quarter'] = df_temp['shipping_date_(DateOrders)'].dt.quarter
        df_temp['shipping_day'] = df_temp['shipping_date_(DateOrders)'].dt.day
    
    temporal_features = ['shipping_month', 'shipping_dayofweek', 'shipping_quarter', 'shipping_day']
    numeric_features.extend(temporal_features)
    
    # Imputar nulos
    imputer = SimpleImputer(strategy='median')
    df_train[numeric_features] = imputer.fit_transform(df_train[numeric_features])
    df_alertar[numeric_features] = imputer.transform(df_alertar[numeric_features])
    
    # Encoding categóricas
    print('   Encoding categóricas...')
    for col in categorical_features:
        df_train[col] = df_train[col].fillna('Desconocido').astype(str)
        df_alertar[col] = df_alertar[col].fillna('Desconocido').astype(str)
    
    encoders = {}
    for col in categorical_features:
        le = LabelEncoder()
        le.fit(df_train[col])
        df_train[col + '_encoded'] = le.transform(df_train[col])
        
        # Manejar categorías no vistas
        test_values_safe = df_alertar[col].copy()
        unseen_mask = ~test_values_safe.isin(le.classes_)
        if unseen_mask.sum() > 0:
            test_values_safe[unseen_mask] = le.classes_[0]
        df_alertar[col + '_encoded'] = le.transform(test_values_safe)
        df_alertar.loc[unseen_mask, col + '_encoded'] = -1
        
        encoders[col] = le
    
    categorical_features_encoded = [f + '_encoded' for f in categorical_features]
    features_final = numeric_features + categorical_features_encoded
    
    # Preparar matrices
    X_train = df_train[features_final].fillna(0).copy()
    y_train = df_train['Late_delivery_risk'].copy()
    X_alertar = df_alertar[features_final].fillna(0).copy()
    
    print(f'\n✓ Datos preparados para entrenamiento')
    print(f'   X_train: {X_train.shape}')
    print(f'   X_alertar: {X_alertar.shape}')
    
    # Entrenar LightGBM
    print(f'\n🤖 Entrenando LightGBM...')
    
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'random_state': 42
    }
    
    train_data = lgb.Dataset(X_train, label=y_train)
    model = lgb.train(
        params,
        train_data,
        num_boost_round=200,
        valid_sets=[train_data],
        valid_names=['train']
    )
    
    print(f'\n✓ Modelo entrenado: {model.num_trees()} árboles')
    
    # Predecir scores
    print(f'\n🎯 Generando scores de riesgo...')
    df_alertar['score_model'] = model.predict(X_alertar, num_iteration=model.best_iteration)
    
    print(f'✓ Scores generados:')
    print(f'   Min:    {df_alertar["score_model"].min():.4f}')
    print(f'   Max:    {df_alertar["score_model"].max():.4f}')
    print(f'   Media:  {df_alertar["score_model"].mean():.4f}')
    print(f'   Mediana: {df_alertar["score_model"].median():.4f}')

elif not model_exists and modo == "PRODUCCIÓN":
    print('\n❌ ERROR: Modo producción requiere modelo guardado')
    print('   No se puede re-entrenar sin datos históricos de train')
    raise ValueError('Modelo no disponible para modo producción')


=== RE-ENTRENAMIENTO DEL MODELO 1 ===

📋 Features del modelo:
   Numéricas: 5
   Categóricas: 9
   Encoding categóricas...

✓ Datos preparados para entrenamiento
   X_train: (138212, 18)
   X_alertar: (34553, 18)

🤖 Entrenando LightGBM...

✓ Modelo entrenado: 200 árboles

🎯 Generando scores de riesgo...
✓ Scores generados:
   Min:    0.0001
   Max:    1.0000
   Media:  0.5698
   Mediana: 0.9998


---

## SECCIÓN 4 — GENERACIÓN DE ALERTAS

Aplicar política configurada para marcar alertas (threshold o top-k).

---

### Aplicar Política de Alertas

Marcar como alerta según la política configurada: umbral fijo o cupo top-k.

In [8]:
print('='*70)
print(' '*15 + 'APLICACIÓN DE POLÍTICA DE ALERTAS')
print('='*70)

print(f'\n📊 Política seleccionada: {alert_policy.upper()}')

if alert_policy == "threshold":
    # Política 1: Umbral fijo
    df_alertar['alert_flag'] = (df_alertar['score_model'] >= threshold_value).astype(int)
    
    n_alertas = df_alertar['alert_flag'].sum()
    pct_alertas = n_alertas / len(df_alertar) * 100
    
    print(f'\n✓ Umbral aplicado: {threshold_value}')
    print(f'  Alertas generadas: {n_alertas:,} ({pct_alertas:.2f}% del total)')
    
elif alert_policy == "topk":
    # Política 2: Top-k (cupo fijo)
    threshold_topk = np.percentile(df_alertar['score_model'], 100 - topk_percent)
    df_alertar['alert_flag'] = (df_alertar['score_model'] >= threshold_topk).astype(int)
    
    n_alertas = df_alertar['alert_flag'].sum()
    pct_alertas = n_alertas / len(df_alertar) * 100
    
    print(f'\n✓ Cupo aplicado: Top {topk_percent}%')
    print(f'  Umbral resultante: {threshold_topk:.4f}')
    print(f'  Alertas generadas: {n_alertas:,} ({pct_alertas:.2f}% del total)')

else:
    raise ValueError(f'Política no válida: {alert_policy}')

# Estadísticas de scores en alertas vs no alertas
scores_alertas = df_alertar[df_alertar['alert_flag'] == 1]['score_model']
scores_no_alertas = df_alertar[df_alertar['alert_flag'] == 0]['score_model']

print(f'\n📈 Distribución de scores:')
print(f'   Alertas (flag=1): mean={scores_alertas.mean():.4f}, min={scores_alertas.min():.4f}, max={scores_alertas.max():.4f}')
print(f'   No alertas (flag=0): mean={scores_no_alertas.mean():.4f}, min={scores_no_alertas.min():.4f}, max={scores_no_alertas.max():.4f}')

               APLICACIÓN DE POLÍTICA DE ALERTAS

📊 Política seleccionada: TOPK

✓ Cupo aplicado: Top 20%
  Umbral resultante: 0.9999
  Alertas generadas: 6,912 (20.00% del total)

📈 Distribución de scores:
   Alertas (flag=1): mean=0.9999, min=0.9999, max=1.0000
   No alertas (flag=0): mean=0.4622, min=0.0001, max=0.9999


### Validación en Simulación (Solo si existe Late_delivery_risk)

Si estamos en modo simulación y existe el target real, calcular precision/recall como validación.

In [9]:
if modo == "SIMULACIÓN" and 'Late_delivery_risk' in df_alertar.columns:
    print('\n=== VALIDACIÓN EN SIMULACIÓN ===\n')
    print('⚠️ NOTA: En producción real NO tenemos Late_delivery_risk al despacho.')
    print('   Este cálculo es solo para validar el sistema en datos históricos.\n')
    
    y_true = df_alertar['Late_delivery_risk'].values
    y_pred = df_alertar['alert_flag'].values
    
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    tn = ((y_pred == 0) & (y_true == 0)).sum()
    
    print(f'📊 Métricas de validación:')
    print(f'   Precision: {precision:.4f} (de las alertas, {precision*100:.1f}% son tardanzas reales)')
    print(f'   Recall:    {recall:.4f} (detectamos {recall*100:.1f}% de las tardanzas reales)')
    print(f'\n   True Positives:  {tp:,} (alertas correctas)')
    print(f'   False Positives: {fp:,} (falsa alarma)')
    print(f'   False Negatives: {fn:,} (tardanzas no detectadas)')
    print(f'   True Negatives:  {tn:,} (correctamente no alertados)')
    
    print(f'\n💡 Interpretación operativa:')
    print(f'   → De cada 100 alertas, ~{int(precision*100)} son tardanzas reales')
    print(f'   → Detectamos ~{int(recall*100)} de cada 100 tardanzas que ocurren')


=== VALIDACIÓN EN SIMULACIÓN ===

⚠️ NOTA: En producción real NO tenemos Late_delivery_risk al despacho.
   Este cálculo es solo para validar el sistema en datos históricos.

📊 Métricas de validación:
   Precision: 1.0000 (de las alertas, 100.0% son tardanzas reales)
   Recall:    0.3483 (detectamos 34.8% de las tardanzas reales)

   True Positives:  6,912 (alertas correctas)
   False Positives: 0 (falsa alarma)
   False Negatives: 12,935 (tardanzas no detectadas)
   True Negatives:  14,706 (correctamente no alertados)

💡 Interpretación operativa:
   → De cada 100 alertas, ~100 son tardanzas reales
   → Detectamos ~34 de cada 100 tardanzas que ocurren


### Preparar DataFrame de Alertas

Crear estructura final con las columnas necesarias para operación.

In [10]:
print('\n=== PREPARACIÓN DE ALERTAS PARA EXPORT ===\n')

# Columnas clave para alertas
cols_alertas = ['row_id', 'shipping_date_(DateOrders)', 'score_model', 'alert_flag']

# Agregar Order_Id si existe
if 'Order_Id' in df_alertar.columns:
    cols_alertas.insert(1, 'Order_Id')
    print('✓ Order_Id incluido en alertas')

# Agregar contexto operativo
cols_contexto = ['Shipping_Mode', 'Order_Region', 'Market', 'Order_Country', 'Order_State']
cols_contexto = [c for c in cols_contexto if c in df_alertar.columns]
cols_alertas.extend(cols_contexto)

# Crear dataframe de alertas (solo las filas con alert_flag=1)
df_alertas_only = df_alertar[df_alertar['alert_flag'] == 1][cols_alertas].copy()
df_alertas_only = df_alertas_only.sort_values('score_model', ascending=False)

print(f'✓ Alertas preparadas: {len(df_alertas_only):,} registros')
print(f'  Columnas: {list(df_alertas_only.columns)}')

# Crear también dataframe completo con scoring (para auditoría)
df_scoring_completo = df_alertar[cols_alertas].copy()
df_scoring_completo = df_scoring_completo.sort_values('score_model', ascending=False)

print(f'\n✓ Scoring completo preparado: {len(df_scoring_completo):,} registros')
print(f'  (incluye alertas + no alertas)')


=== PREPARACIÓN DE ALERTAS PARA EXPORT ===

✓ Order_Id incluido en alertas
✓ Alertas preparadas: 6,912 registros
  Columnas: ['row_id', 'Order_Id', 'shipping_date_(DateOrders)', 'score_model', 'alert_flag', 'Shipping_Mode', 'Order_Region', 'Market', 'Order_Country', 'Order_State']

✓ Scoring completo preparado: 34,553 registros
  (incluye alertas + no alertas)


---

## SECCIÓN 5 — SALIDAS Y RESUMEN

Exportar archivos de alertas, scoring completo y resumen ejecutivo.

---

### Guardar Archivos CSV

Exportar alertas y scoring completo a CSV.

In [11]:
print('='*70)
print(' '*15 + 'EXPORT DE ARCHIVOS')
print('='*70)

output_dir = data_dir

# 1) Guardar ALERTAS (solo flag=1)
alertas_path = output_dir / output_alerts_file
df_alertas_only.to_csv(alertas_path, index=False)

print(f'\n✓ Alertas guardadas: {alertas_path}')
print(f'  Tamaño: {alertas_path.stat().st_size / 1024:.2f} KB')
print(f'  Filas: {len(df_alertas_only):,}')

# 2) Guardar SCORING COMPLETO (todas las filas)
scoring_path = output_dir / output_scoring_full
df_scoring_completo.to_csv(scoring_path, index=False)

print(f'\n✓ Scoring completo guardado: {scoring_path}')
print(f'  Tamaño: {scoring_path.stat().st_size / 1024:.2f} KB')
print(f'  Filas: {len(df_scoring_completo):,}')

               EXPORT DE ARCHIVOS

✓ Alertas guardadas: ..\data\processed\alertas.csv
  Tamaño: 790.92 KB
  Filas: 6,912

✓ Scoring completo guardado: ..\data\processed\scoring_completo.csv
  Tamaño: 4054.70 KB
  Filas: 34,553


### Generar Resumen Agregado

Calcular estadísticas clave y segmentos con más alertas.

In [12]:
print('\n=== RESUMEN AGREGADO ===\n')

# Estadísticas generales
total_envios = len(df_alertar)
total_alertas = len(df_alertas_only)
pct_alertas_final = total_alertas / total_envios * 100
score_promedio_alertas = df_alertas_only['score_model'].mean()

print(f'📊 ESTADÍSTICAS GENERALES:')
print(f'   Total envíos evaluados: {total_envios:,}')
print(f'   Total alertas: {total_alertas:,} ({pct_alertas_final:.2f}%)')
print(f'   Score promedio alertas: {score_promedio_alertas:.4f}')

# Top combinaciones con más alertas
if 'Shipping_Mode' in df_alertas_only.columns and 'Order_Region' in df_alertas_only.columns:
    top_combos = df_alertas_only.groupby(['Shipping_Mode', 'Order_Region']).agg(
        n_alertas=('row_id', 'count'),
        score_promedio=('score_model', 'mean')
    ).reset_index().sort_values('n_alertas', ascending=False).head(10)
    
    print(f'\n📍 TOP 10 COMBINACIONES (Shipping_Mode, Order_Region) CON MÁS ALERTAS:')
    print(top_combos.to_string(index=False))
    
    # Guardar para resumen
    top_combos_text = top_combos.to_string(index=False)
else:
    top_combos_text = 'No disponible (columnas faltantes)'
    print(f'\n⚠️ No se pudieron calcular combinaciones (columnas faltantes)')

# Alertas por Market (si existe)
if 'Market' in df_alertas_only.columns:
    alertas_por_market = df_alertas_only['Market'].value_counts()
    print(f'\n🌍 ALERTAS POR MARKET:')
    print(alertas_por_market)
    alertas_market_text = alertas_por_market.to_string()
else:
    alertas_market_text = 'No disponible'


=== RESUMEN AGREGADO ===

📊 ESTADÍSTICAS GENERALES:
   Total envíos evaluados: 34,553
   Total alertas: 6,912 (20.00%)
   Score promedio alertas: 0.9999

📍 TOP 10 COMBINACIONES (Shipping_Mode, Order_Region) CON MÁS ALERTAS:
Shipping_Mode    Order_Region  n_alertas  score_promedio
  First Class  Western Europe       1844          0.9999
  First Class Central America        805          0.9999
  First Class Southern Europe        635          0.9999
 Second Class  Western Europe        557          0.9999
  First Class Northern Europe        521          0.9999
  First Class   South America        414          0.9999
 Second Class Central America        276          0.9999
  First Class  Southeast Asia        241          0.9999
  First Class         Oceania        208          0.9999
 Second Class Northern Europe        197          0.9999

🌍 ALERTAS POR MARKET:
Market
Europe          3933
LATAM           1869
Pacific Asia    1110
Name: count, dtype: int64


### Guardar Resumen Ejecutivo (TXT)

Crear archivo de texto con resumen para rápida revisión operativa.

In [13]:
# Crear resumen ejecutivo
resumen_text = f"""
================================================================================
               RESUMEN EJECUTIVO - SISTEMA DE ALERTAS
              Riesgo de Tardanza al Momento del Despacho
================================================================================

FECHA DE EJECUCIÓN: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
MODO: {modo}

CONFIGURACIÓN
-------------
Política: {alert_policy.upper()}
"""

if alert_policy == "threshold":
    resumen_text += f"Umbral: {threshold_value}\n"
elif alert_policy == "topk":
    resumen_text += f"Cupo: Top {topk_percent}%\n"

resumen_text += f"""
RESULTADOS
----------
Total envíos evaluados: {total_envios:,}
Total alertas generadas: {total_alertas:,} ({pct_alertas_final:.1f}% del total)
Score promedio alertas: {score_promedio_alertas:.4f}

TOP 10 COMBINACIONES CON MÁS ALERTAS
-------------------------------------
{top_combos_text}

"""

if modo == "SIMULACIÓN" and 'Late_delivery_risk' in df_alertar.columns:
    resumen_text += f"""
VALIDACIÓN EN SIMULACIÓN
-------------------------
(Solo disponible con datos históricos)

Precision: {precision:.4f} ({precision*100:.1f}% de alertas son tardanzas reales)
Recall:    {recall:.4f} (detectamos {recall*100:.1f}% de tardanzas)

True Positives:  {tp:,}
False Positives: {fp:,}
False Negatives: {fn:,}
True Negatives:  {tn:,}

"""

resumen_text += """
================================================================================
                    CÓMO USAR ESTE SISTEMA EN OPERACIÓN
================================================================================

1. REVISAR ALERTAS DIARIAS
   - Abrir archivo: alertas.csv
   - Priorizar por score_model (mayor = mayor riesgo)
   
2. ACCIÓN SUGERIDA ANTE UNA ALERTA
   a) Score >= 0.7: ALTA PRIORIDAD
      → Revisión manual inmediata
      → Contacto preventivo con cliente
      → Considerar cambio de método de envío
   
   b) Score 0.5-0.7: PRIORIDAD MEDIA
      → Seguimiento cercano
      → Notificación al equipo de logística
   
   c) Score < 0.5: MONITOREO
      → Seguimiento estándar
      
3. SEGMENTOS DE ALTO RIESGO
   - Revisar "TOP 10 COMBINACIONES" arriba
   - Focalizar mejoras operativas en estos segmentos
   
4. AUDITORÍA
   - Archivo scoring_completo.csv contiene TODOS los envíos con score
   - Útil para análisis post-mortem y mejora continua

================================================================================
"""

# Guardar resumen
summary_path = output_dir / output_summary_file
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(resumen_text)

print(f'\n✓ Resumen ejecutivo guardado: {summary_path}')
print(f'  Tamaño: {summary_path.stat().st_size / 1024:.2f} KB')


✓ Resumen ejecutivo guardado: ..\data\processed\resumen_ejecutivo.txt
  Tamaño: 2.61 KB


---

## ✅ SISTEMA DE ALERTAS COMPLETADO

---

In [14]:
print('='*70)
print(' '*10 + '✅ SISTEMA DE ALERTAS COMPLETADO')
print('='*70)

print(f'\n📁 ARCHIVOS GENERADOS:')
print(f'\n1. {output_alerts_file}')
print(f'   → Alertas de alto riesgo (flag=1)')
print(f'   → Usar para acción inmediata')
print(f'   → {len(df_alertas_only):,} alertas')

print(f'\n2. {output_scoring_full}')
print(f'   → Scoring completo de todos los envíos')
print(f'   → Útil para auditoría y análisis')
print(f'   → {len(df_scoring_completo):,} registros')

print(f'\n3. {output_summary_file}')
print(f'   → Resumen ejecutivo en texto plano')
print(f'   → Leer para contexto rápido')

print(f'\n\n💡 PRÓXIMOS PASOS SUGERIDOS:')
print(f'   1. Revisar {output_summary_file} para visión general')
print(f'   2. Abrir {output_alerts_file} y priorizar por score_model')
print(f'   3. Implementar acciones preventivas en alertas de score alto')
print(f'   4. Monitorear tasas de precisión reales (post-entrega)')
print(f'   5. Ajustar política (threshold o topk_percent) según capacidad')

print('\n' + '='*70)

          ✅ SISTEMA DE ALERTAS COMPLETADO

📁 ARCHIVOS GENERADOS:

1. alertas.csv
   → Alertas de alto riesgo (flag=1)
   → Usar para acción inmediata
   → 6,912 alertas

2. scoring_completo.csv
   → Scoring completo de todos los envíos
   → Útil para auditoría y análisis
   → 34,553 registros

3. resumen_ejecutivo.txt
   → Resumen ejecutivo en texto plano
   → Leer para contexto rápido


💡 PRÓXIMOS PASOS SUGERIDOS:
   1. Revisar resumen_ejecutivo.txt para visión general
   2. Abrir alertas.csv y priorizar por score_model
   3. Implementar acciones preventivas en alertas de score alto
   4. Monitorear tasas de precisión reales (post-entrega)
   5. Ajustar política (threshold o topk_percent) según capacidad



---

### Visualización Rápida de Alertas (Opcional)

Top 20 alertas de mayor riesgo para revisión inmediata.

In [15]:
print('='*70)
print(' '*10 + 'TOP 20 ALERTAS DE MAYOR RIESGO')
print('='*70)

# Mostrar top 20 alertas
cols_display = ['shipping_date_(DateOrders)', 'score_model', 'Shipping_Mode', 'Order_Region', 'Market']
cols_display = [c for c in cols_display if c in df_alertas_only.columns]

top_20_alertas = df_alertas_only[cols_display].head(20)

print(f'\nTop 20 alertas ordenadas por score (mayor riesgo primero):\n')
print(top_20_alertas.to_string(index=False))

print(f'\n💡 Estas son las alertas más urgentes para revisión.')
print(f'   Considerar acción preventiva inmediata.')

          TOP 20 ALERTAS DE MAYOR RIESGO

Top 20 alertas ordenadas por score (mayor riesgo primero):

shipping_date_(DateOrders)  score_model Shipping_Mode    Order_Region       Market
       2017-10-11 19:36:00       1.0000   First Class  Western Europe       Europe
       2017-10-11 18:12:00       1.0000   First Class  Western Europe       Europe
       2017-06-15 09:36:00       1.0000   First Class   South America        LATAM
       2017-06-15 09:36:00       1.0000   First Class   South America        LATAM
       2017-05-18 20:51:00       1.0000   First Class   South America        LATAM
       2017-10-14 14:52:00       1.0000   First Class  Western Europe       Europe
       2017-06-29 17:17:00       1.0000   First Class  Western Europe       Europe
       2017-06-29 17:17:00       1.0000   First Class  Western Europe       Europe
       2017-10-26 11:21:00       1.0000   First Class  Western Europe       Europe
       2017-06-29 17:17:00       1.0000   First Class  Western Europ